In [ ]:
# ============================================================
# B7 pseudo + B6 pseudo + ConvNeXt pseudo ensemble
# With:
#   - no old matrix path
#   - species bonus sweep: 0.00, 0.10
#   - KNN weight sweep: 0.50, 0.60
#   - proto_mode = "single"
#   - newratio = 0.200
# ============================================================
!pip install -q huggingface_hub

import gc
import os

import numpy as np
import pandas as pd
import torch
from huggingface_hub import (
    create_repo,
    hf_hub_download,
    login,
    upload_file,
)
from tqdm.auto import tqdm

# ============================================================
# hf_token
# ============================================================
HF_OUTPUT_REPO_ID = "liu-peilin/happywhale_ensemble_v2xl_b6_b7_conv_v1"
HF_REPO_TYPE = "model"


def get_hf_token_required():
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")

    if token:
        return token

    try:
        from google.colab import userdata

        token = userdata.get("HF_TOKEN")
        if token:
            os.environ["HF_TOKEN"] = token
            return token
    except Exception:
        pass

    from getpass import getpass

    token = getpass("Paste Hugging Face token with write permission: ")
    os.environ["HF_TOKEN"] = token
    return token


HF_TOKEN = get_hf_token_required()
login(token=HF_TOKEN, add_to_git_credential=False)

create_repo(
    repo_id=HF_OUTPUT_REPO_ID,
    repo_type=HF_REPO_TYPE,
    private=True,
    exist_ok=True,
    token=HF_TOKEN,
)
HF_TOKEN = get_hf_token_required()
login(token=HF_TOKEN, add_to_git_credential=False)
print("HF output repo ready:", HF_OUTPUT_REPO_ID)


def get_hf_token_optional():
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")

    if token:
        return token

    try:
        from google.colab import userdata

        token = userdata.get("HF_TOKEN")
        if token:
            os.environ["HF_TOKEN"] = token
            return token
    except Exception:
        pass

    return None


def download_hf_file_if_missing(repo_id, hf_path, local_path):
    if os.path.exists(local_path):
        return local_path

    os.makedirs(os.path.dirname(local_path), exist_ok=True)

    token = get_hf_token_optional()

    print("Downloading from HF:")
    print(" repo:", repo_id)
    print(" file:", hf_path)
    print(" ->", local_path)

    downloaded = hf_hub_download(
        repo_id=repo_id,
        repo_type=HF_REPO_TYPE,
        filename=hf_path,
        token=token,
    )

    import shutil

    shutil.copy2(downloaded, local_path)
    return local_path


def download_model_mat_from_hf_if_missing(local_path):
    """
    Download one model-level matrix/test_images file from HF output repo.
    """
    if os.path.exists(local_path):
        print("Local model mat exists, skip HF download:", local_path)
        return local_path

    hf_path = f"{HF_MODEL_MAT_DIR}/{os.path.basename(local_path)}"

    print("Downloading model mat from HF:")
    print(" repo:", HF_OUTPUT_REPO_ID)
    print(" file:", hf_path)
    print(" ->", local_path)

    downloaded = hf_hub_download(
        repo_id=HF_OUTPUT_REPO_ID,
        repo_type=HF_REPO_TYPE,
        filename=hf_path,
        token=HF_TOKEN,
    )

    import shutil

    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    shutil.copy2(downloaded, local_path)

    return local_path


def upload_model_mat_to_hf_if_enabled(local_path):
    """
    Upload one model-level matrix/test_images file to HF output repo.
    """
    if not UPLOAD_MODEL_MATS_TO_HF:
        return

    if not os.path.exists(local_path):
        raise FileNotFoundError(f"Cannot upload missing file: {local_path}")

    hf_path = f"{HF_MODEL_MAT_DIR}/{os.path.basename(local_path)}"

    print("Uploading model mat to HF:")
    print(" local:", local_path)
    print(" repo:", HF_OUTPUT_REPO_ID)
    print(" path:", hf_path)

    upload_file(
        path_or_fileobj=local_path,
        path_in_repo=hf_path,
        repo_id=HF_OUTPUT_REPO_ID,
        repo_type=HF_REPO_TYPE,
        token=HF_TOKEN,
    )

    print("Uploaded:", hf_path)


# ============================================================
# Paths
# Hugging Face input repositories
# ============================================================


HF_REPO_ID_B6 = os.environ.get(
    "HF_REPO_ID_B6",
    "liu-peilin/happywhale_b6_pseudo_charm_round2_1024_v2",
)
HF_REPO_ID_CONV = os.environ.get(
    "HF_REPO_ID_CONV",
    "liu-peilin/conv",
)

HF_REPO_ID_V2XL = os.environ.get(
    "HF_REPO_ID_V2XL",
    "fangfang777/happywhale_v2xl_pseudo_768",
)
HF_REPO_ID_B7 = os.environ.get(
    "HF_REPO_ID_B7",
    "fangfang777/happywhale_b7_multicrop_fnb_subcenter_species_pseudo_1024_by_lin",
)

# Local cache under Colab /content, not Google Drive
LOCAL_INPUT_ROOT = "/content/hf_inputs"
OUT_DIR = "/content/ensemble_b7_b6_v2xl_conv_outputs"
CACHE_DIR = "/content/ensemble_b7_b6_v2xl_conv_cache"

os.makedirs(LOCAL_INPUT_ROOT, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_SPECS = {
    "v2xl": {
        "repo_id": HF_REPO_ID_V2XL,
        "hf_dir": "embeddings",
        "local_dir": os.path.join(LOCAL_INPUT_ROOT, "v2_xl"),
    },
    "b6": {
        "repo_id": HF_REPO_ID_B6,
        "hf_dir": "embeddings",
        "local_dir": os.path.join(LOCAL_INPUT_ROOT, "b6"),
    },
    "b7": {
        "repo_id": HF_REPO_ID_B7,
        "hf_dir": "embeddings_31",
        "local_dir": os.path.join(LOCAL_INPUT_ROOT, "b7"),
    },
    "conv": {
        "repo_id": HF_REPO_ID_CONV,
        "hf_dir": "",
        "local_dir": os.path.join(LOCAL_INPUT_ROOT, "conv"),
    },
}

for tag, spec in MODEL_SPECS.items():
    os.makedirs(spec["local_dir"], exist_ok=True)

# ============================================================
# Model matrix cache mode
# ============================================================

USE_HF_MODEL_MATS = True
# True:
#   download model_mats/*.npy from Hugging Face and only sweep model weights / newratio.
# False:
#   rebuild model matrices from train/test npz, KNN, logits, proto.

UPLOAD_MODEL_MATS_TO_HF = False
# Only used when USE_HF_MODEL_MATS = False.
# After rebuilding model_mat.npy, upload it to HF for future reuse.

HF_MODEL_MAT_DIR = "model_mats"

# ============================================================
# Settings
# ============================================================
LABEL_CLASSES_REPO_ID = HF_REPO_ID_B6
LABEL_CLASSES_HF_PATH = "metadata/label_classes.npy"

LABEL_CLASSES_PATH = os.path.join(
    LOCAL_INPUT_ROOT,
    "metadata",
    "label_classes.npy",
)

download_hf_file_if_missing(
    repo_id=LABEL_CLASSES_REPO_ID,
    hf_path=LABEL_CLASSES_HF_PATH,
    local_path=LABEL_CLASSES_PATH,
)

label_classes = np.load(LABEL_CLASSES_PATH, allow_pickle=True)
NUM_CLASSES = len(label_classes)

print("LABEL_CLASSES_REPO_ID:", LABEL_CLASSES_REPO_ID)
print("LABEL_CLASSES_HF_PATH:", LABEL_CLASSES_HF_PATH)
print("LABEL_CLASSES_PATH:", LABEL_CLASSES_PATH)
print("NUM_CLASSES:", NUM_CLASSES)

NEW_RATIOS = [0.165, 0.180, 0.200, 0.215]

# KNN/prototype settings
KNN_NEIGHBORS = 500
KNN_CHUNK_SIZE = 128

# # ============================================================
# # Proto=0 baseline settings
# # ============================================================
# USE_PROTO_SCORE = False
# PROTO_MODE = "none"
# PROTO_WEIGHT = 0.00

# KNN_WEIGHT_LIST = [0.50]
# SPECIES_BONUS_LIST = [0.00, 0.10]
# proto_mode settings
USE_PROTO_SCORE = True
PROTO_MODE = "single"

FUSION_WEIGHT_SETS = [
    {
        "name": "knn050_logit025_proto025",
        "knn": 0.50,
        "logit": 0.25,
        "proto": 0.25,
    },
]

SPECIES_BONUS_LIST = [0.00, 0.10]
SPECIES_USE_TOPK = 1

# If True, rebuild matrices even if cached in this new CACHE_DIR.
# First run can be False because CACHE_DIR is new.
# If you change logic and want to rebuild, set True.
FORCE_REBUILD = False

print("FUSION_WEIGHT_SETS:", FUSION_WEIGHT_SETS)
print("SPECIES_BONUS_LIST:", SPECIES_BONUS_LIST)
print("USE_PROTO_SCORE:", USE_PROTO_SCORE)
print("PROTO_MODE:", PROTO_MODE)

# ============================================================
# Crop weights
# ============================================================

# V2-XL
V2XL_CROP_WEIGHTS = {
    "fullbody": 0.80,
    "backfin": 0.10,
    "none": 0.10,
}

# B6 charm best direction
B6_CROP_WEIGHTS = {
    "fullbody": 0.60,
    "fullbody_charm": 0.25,
    "backfin": 0.05,
    "none": 0.10,
}

# B7 best
B7_CROP_WEIGHTS = {
    "fullbody": 0.85,
    "backfin": 0.05,
    "none": 0.10,
}

# Conv
CONV_CROP_WEIGHTS = {
    "fullbody": 0.85,
    "backfin": 0.05,
    "none": 0.10,
}


# ============================================================
# Model-level weights to try
# ============================================================

MODEL_WEIGHT_SETS = [
    {
        "name": "v2xl_010_b7_070_b6_015_conv_005",
        "v2xl": 0.10,
        "b7": 0.70,
        "b6": 0.15,
        "conv": 0.05,
    },
    {
        "name": "v2xl_015_b7_065_b6_015_conv_005",
        "v2xl": 0.15,
        "b7": 0.65,
        "b6": 0.15,
        "conv": 0.05,
    },
    {
        "name": "v2xl_020_b7_060_b6_015_conv_005",
        "v2xl": 0.20,
        "b7": 0.60,
        "b6": 0.15,
        "conv": 0.05,
    },
    {
        "name": "v2xl_010_b7_065_b6_020_conv_005",
        "v2xl": 0.10,
        "b7": 0.65,
        "b6": 0.20,
        "conv": 0.05,
    },
    {
        "name": "v2xl_000_b7_070_b6_020_conv_010",
        "v2xl": 0.00,
        "b7": 0.70,
        "b6": 0.20,
        "conv": 0.10,
    },
]


# ============================================================
# Basic checks
# ============================================================
def hf_join(hf_dir, filename):
    if hf_dir is None or hf_dir == "":
        return filename
    return f"{hf_dir.rstrip('/')}/{filename}"


def download_hf_file_if_missing(repo_id, hf_path, local_path):
    if os.path.exists(local_path):
        return local_path

    os.makedirs(os.path.dirname(local_path), exist_ok=True)

    print("Downloading from HF:")
    print(" repo:", repo_id)
    print(" file:", hf_path)
    print(" ->", local_path)

    downloaded = hf_hub_download(
        repo_id=repo_id,
        repo_type=HF_REPO_TYPE,
        filename=hf_path,
        token=HF_TOKEN,
    )

    import shutil

    shutil.copy2(downloaded, local_path)
    return local_path


def prepare_model_npz_files(model_tag, crop_weights):
    spec = MODEL_SPECS[model_tag]
    repo_id = spec["repo_id"]
    hf_dir = spec["hf_dir"]
    local_dir = spec["local_dir"]

    required_crops = list(crop_weights.keys())

    for crop in required_crops:
        for split in ["train", "test"]:
            filename = f"{split}_{crop}_results.npz"
            hf_path = hf_join(hf_dir, filename)
            local_path = os.path.join(local_dir, filename)

            download_hf_file_if_missing(
                repo_id=repo_id,
                hf_path=hf_path,
                local_path=local_path,
            )

    return local_dir


def check_required_npz(model_name, model_dir, crop_weights):
    print(f"\nChecking {model_name}: {model_dir}")

    required = []
    for crop in crop_weights.keys():
        required.append(os.path.join(model_dir, f"train_{crop}_results.npz"))
        required.append(os.path.join(model_dir, f"test_{crop}_results.npz"))

    missing = [p for p in required if not os.path.exists(p)]

    if missing:
        print("Missing files:")
        for p in missing:
            print(" ", p)
        raise FileNotFoundError(f"{model_name} missing npz files.")

    for p in required:
        print("OK:", p)


def check_npz_class_consistency(model_name, model_dir):
    p = os.path.join(model_dir, "test_fullbody_results.npz")
    z = np.load(p, allow_pickle=True)

    print(f"\n{model_name}")
    print("pred_idx shape:", z["pred_idx"].shape)
    print("pred_logit shape:", z["pred_logit"].shape)

    max_idx = int(z["pred_idx"].max())
    print("max pred_idx:", max_idx)
    print("NUM_CLASSES:", NUM_CLASSES)

    if max_idx >= NUM_CLASSES:
        raise ValueError(
            f"{model_name} pred_idx max {max_idx} >= NUM_CLASSES {NUM_CLASSES}. "
            "Label mapping may be inconsistent."
        )


V2XL_DIR = prepare_model_npz_files("v2xl", V2XL_CROP_WEIGHTS)
B6_DIR = prepare_model_npz_files("b6", B6_CROP_WEIGHTS)
B7_DIR = prepare_model_npz_files("b7", B7_CROP_WEIGHTS)
CONVNEXT_DIR = prepare_model_npz_files("conv", CONV_CROP_WEIGHTS)

print("V2XL_DIR:", V2XL_DIR)
print("B6_DIR:", B6_DIR)
print("B7_DIR:", B7_DIR)
print("CONVNEXT_DIR:", CONVNEXT_DIR)

check_required_npz("V2-XL", V2XL_DIR, V2XL_CROP_WEIGHTS)
check_required_npz("B6", B6_DIR, B6_CROP_WEIGHTS)
check_required_npz("B7", B7_DIR, B7_CROP_WEIGHTS)
check_required_npz("ConvNeXt", CONVNEXT_DIR, CONV_CROP_WEIGHTS)
check_npz_class_consistency("V2XL pseudo", V2XL_DIR)
check_npz_class_consistency("B7 pseudo", B7_DIR)
check_npz_class_consistency("B6 pseudo", B6_DIR)
check_npz_class_consistency("ConvNeXt pseudo", CONVNEXT_DIR)


# ============================================================
# Utility functions
# ============================================================


def normalize_np(x):
    x = x.astype("float32")
    return x / np.linalg.norm(x, axis=1, keepdims=True).clip(min=1e-12)


def restore_all_pred(n_class, pred, pred_idx):
    n_data = pred.shape[0]
    all_pred = np.zeros((n_data, n_class), dtype=np.float32)

    for i in tqdm(range(n_data), desc="restore logits"):
        all_pred[i, pred_idx[i]] = pred[i]

    return all_pred


def knn_all_pred_torch(
    n_class,
    test_feat,
    train_feat,
    train_label,
    n_neighbors=500,
    chunk_size=128,
):
    train_feat = normalize_np(train_feat)
    test_feat = normalize_np(test_feat)

    train_tensor = torch.tensor(train_feat, dtype=torch.float32).to(DEVICE)
    test_tensor = torch.tensor(test_feat, dtype=torch.float32).to(DEVICE)

    out = np.zeros((len(test_feat), n_class), dtype=np.float32)

    with torch.no_grad():
        for start in tqdm(
            range(0, len(test_tensor), chunk_size), desc="KNN all pred"
        ):
            end = min(start + chunk_size, len(test_tensor))
            query = test_tensor[start:end]

            sim = query @ train_tensor.T
            topv, topi = torch.topk(sim, k=n_neighbors, dim=1)

            topv = topv.cpu().numpy()
            topi = topi.cpu().numpy()

            for bi in range(topi.shape[0]):
                labels_i = train_label[topi[bi]]
                scores_i = topv[bi]
                np.maximum.at(out[start + bi], labels_i, scores_i)

    del train_tensor, test_tensor

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return out


def knn_both_feat(
    n_class,
    train_feat1,
    train_feat2,
    test_feat1,
    test_feat2,
    train_label,
):
    train_feat12 = np.concatenate([train_feat1, train_feat2], axis=0)
    train_label12 = np.concatenate([train_label, train_label], axis=0)

    knn1 = knn_all_pred_torch(
        n_class,
        test_feat1,
        train_feat12,
        train_label12,
        n_neighbors=KNN_NEIGHBORS,
        chunk_size=KNN_CHUNK_SIZE,
    )

    knn2 = knn_all_pred_torch(
        n_class,
        test_feat2,
        train_feat12,
        train_label12,
        n_neighbors=KNN_NEIGHBORS,
        chunk_size=KNN_CHUNK_SIZE,
    )

    return (knn1 + knn2) / 2.0


def build_class_prototypes_from_two_feats(
    n_class,
    train_feat1,
    train_feat2,
    train_label,
):
    """
    proto_mode = single:
    For each individual_id class, build one prototype by averaging
    both original and flip embeddings.
    """
    train_feat1 = normalize_np(train_feat1)
    train_feat2 = normalize_np(train_feat2)

    train_feat = np.concatenate([train_feat1, train_feat2], axis=0)
    train_label2 = np.concatenate([train_label, train_label], axis=0)

    dim = train_feat.shape[1]
    prototypes = np.zeros((n_class, dim), dtype=np.float32)
    counts = np.zeros(n_class, dtype=np.int64)

    for f, lab in tqdm(
        zip(train_feat, train_label2),
        total=len(train_label2),
        desc="Build single prototypes",
    ):
        lab = int(lab)
        prototypes[lab] += f
        counts[lab] += 1

    valid_mask = counts > 0
    prototypes[valid_mask] /= counts[valid_mask, None]
    prototypes[valid_mask] = normalize_np(prototypes[valid_mask])

    return prototypes, valid_mask


def prototype_all_pred_torch(
    n_class,
    train_feat1,
    train_feat2,
    test_feat1,
    test_feat2,
    train_label,
    chunk_size=128,
):
    """
    Calculate cosine similarity between test embeddings and class prototypes.
    """
    prototypes, valid_mask = build_class_prototypes_from_two_feats(
        n_class,
        train_feat1,
        train_feat2,
        train_label,
    )

    test_feat1 = normalize_np(test_feat1)
    test_feat2 = normalize_np(test_feat2)

    proto_tensor = torch.tensor(prototypes, dtype=torch.float32).to(DEVICE)
    test_tensor1 = torch.tensor(test_feat1, dtype=torch.float32).to(DEVICE)
    test_tensor2 = torch.tensor(test_feat2, dtype=torch.float32).to(DEVICE)

    out1 = np.zeros((len(test_feat1), n_class), dtype=np.float32)
    out2 = np.zeros((len(test_feat2), n_class), dtype=np.float32)

    with torch.no_grad():
        for start in tqdm(
            range(0, len(test_tensor1), chunk_size),
            desc="Prototype pred feat1",
        ):
            end = min(start + chunk_size, len(test_tensor1))
            sim = test_tensor1[start:end] @ proto_tensor.T
            out1[start:end] = sim.cpu().numpy()

        for start in tqdm(
            range(0, len(test_tensor2), chunk_size),
            desc="Prototype pred feat2",
        ):
            end = min(start + chunk_size, len(test_tensor2))
            sim = test_tensor2[start:end] @ proto_tensor.T
            out2[start:end] = sim.cpu().numpy()

    proto_mat = (out1 + out2) / 2.0

    # Classes without training samples should not be selected.
    proto_mat[:, ~valid_mask] = -1e9

    del proto_tensor, test_tensor1, test_tensor2
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return proto_mat


def build_class_species_map(n_class, train_label, train_species_label):
    class_species = np.full(n_class, -1, dtype=np.int32)

    for cls, sp in zip(train_label, train_species_label):
        cls = int(cls)
        sp = int(sp)

        if class_species[cls] == -1:
            class_species[cls] = sp

    return class_species


def apply_species_bonus_to_mat(
    mat,
    train_results,
    test_results,
    n_class,
    species_bonus,
):
    """
    Add species bonus to individual classes whose species matches
    the test image's predicted species.
    """
    if species_bonus == 0:
        return mat

    if "label_species" not in train_results.files:
        print(
            "WARNING: train_results has no label_species. Skip species bonus."
        )
        return mat

    if "pred_species_idx" not in test_results.files:
        print(
            "WARNING: test_results has no pred_species_idx. Skip species bonus."
        )
        return mat

    print(f"Applying species bonus: {species_bonus}")

    train_label = train_results["label"]
    train_species_label = train_results["label_species"]

    class_species = build_class_species_map(
        n_class=n_class,
        train_label=train_label,
        train_species_label=train_species_label,
    )

    species_to_classes = {}
    valid_classes = np.where(class_species >= 0)[0]

    for cls_idx in valid_classes:
        sp = int(class_species[cls_idx])
        if sp not in species_to_classes:
            species_to_classes[sp] = []
        species_to_classes[sp].append(cls_idx)

    for sp in species_to_classes:
        species_to_classes[sp] = np.array(
            species_to_classes[sp], dtype=np.int64
        )

    out = mat.astype("float32", copy=True)

    pred_species_idx = test_results["pred_species_idx"]
    use_topk = min(SPECIES_USE_TOPK, pred_species_idx.shape[1])

    for i in tqdm(range(out.shape[0]), desc="Apply species bonus"):
        species_candidates = pred_species_idx[i, :use_topk]

        for rank, sp in enumerate(species_candidates):
            sp = int(sp)

            if sp not in species_to_classes:
                continue

            cls_indices = species_to_classes[sp]
            bonus = species_bonus / (rank + 1)
            out[i, cls_indices] += bonus

    return out


def binary_search_threshold_from_rowmax(row_max, new_ratio):
    ok = 0.0
    ng = max(1.5, float(row_max.max()) + 0.1)

    for _ in range(40):
        mid = (ok + ng) / 2
        ratio = np.mean(row_max < mid)

        if ratio <= new_ratio:
            ok = mid
        else:
            ng = mid

    return ok


def make_submission_from_matrix(mat, test_images, out_prefix):
    """
    Memory-safe top5 submission.
    """
    n, c = mat.shape

    row_max = mat.max(axis=1)

    topk = 5
    top_idx = np.argpartition(-mat, kth=topk - 1, axis=1)[:, :topk]
    top_scores = np.take_along_axis(mat, top_idx, axis=1)

    order = np.argsort(-top_scores, axis=1)
    top_idx = np.take_along_axis(top_idx, order, axis=1)
    top_scores = np.take_along_axis(top_scores, order, axis=1)

    for new_ratio in NEW_RATIOS:
        threshold = binary_search_threshold_from_rowmax(row_max, new_ratio)
        print(
            f"{out_prefix} | new_ratio={new_ratio:.3f}, threshold={threshold:.6f}"
        )

        preds_all = []

        for i in tqdm(
            range(n), desc=f"Build CSV {out_prefix} nr={new_ratio:.3f}"
        ):
            candidates = []

            for j in range(topk):
                cls_idx = int(top_idx[i, j])
                score = float(top_scores[i, j])
                candidates.append((str(label_classes[cls_idx]), score))

            candidates.append(("new_individual", float(threshold)))
            candidates = sorted(candidates, key=lambda x: x[1], reverse=True)

            preds_all.append(" ".join([x[0] for x in candidates[:5]]))

        submission = pd.DataFrame(
            {
                "image": test_images,
                "predictions": preds_all,
            }
        )

        out_path = os.path.join(
            OUT_DIR,
            f"{out_prefix}_newratio{new_ratio:.3f}_th{threshold:.6f}.csv",
        )

        submission.to_csv(out_path, index=False)
        print("Saved:", out_path)
        print(submission.head())
        upload_file(
            path_or_fileobj=out_path,
            path_in_repo=f"submissions/{os.path.basename(out_path)}",
            repo_id=HF_OUTPUT_REPO_ID,
            repo_type=HF_REPO_TYPE,
            token=HF_TOKEN,
        )

        print("Uploaded to HF:", f"submissions/{os.path.basename(out_path)}")


# ============================================================
# Load / build crop matrices from npz only
# ============================================================


def model_dir_from_tag(model_tag):
    if model_tag == "v2xl":
        return V2XL_DIR
    if model_tag == "b7":
        return B7_DIR
    if model_tag == "b6":
        return B6_DIR
    if model_tag == "conv":
        return CONVNEXT_DIR
    raise ValueError(model_tag)


def build_or_load_crop_matrix(
    model_tag,
    crop_mode,
    fusion,
    species_bonus,
):
    knn_weight = fusion["knn"]
    logit_weight = fusion["logit"]
    proto_weight = fusion["proto"]
    fusion_name = fusion["name"]

    assert abs(knn_weight + logit_weight + proto_weight - 1.0) < 1e-6

    if USE_PROTO_SCORE:
        if logit_weight < 0:
            raise ValueError(
                f"Invalid weights: knn={knn_weight}, logit={logit_weight}, proto={proto_weight}"
            )
        cache_name = (
            f"{model_tag}_{crop_mode}"
            f"_{fusion_name}"
            f"_species{species_bonus:.2f}_singleproto_mat.npy"
        )
    else:
        logit_weight = 1.0 - knn_weight
        cache_name = (
            f"{model_tag}_{crop_mode}"
            f"_knn{knn_weight:.2f}_logit{logit_weight:.2f}"
            f"_species{species_bonus:.2f}_mat.npy"
        )

    cache_img_name = cache_name.replace("_mat.npy", "_test_images.npy")

    cache_mat_path = os.path.join(CACHE_DIR, cache_name)
    cache_img_path = os.path.join(CACHE_DIR, cache_img_name)

    if (
        (not FORCE_REBUILD)
        and os.path.exists(cache_mat_path)
        and os.path.exists(cache_img_path)
    ):
        print("Loading cached matrix:", cache_mat_path)
        return (
            np.load(cache_mat_path).astype("float32"),
            np.load(cache_img_path, allow_pickle=True),
        )

    model_dir = model_dir_from_tag(model_tag)

    train_path = os.path.join(model_dir, f"train_{crop_mode}_results.npz")
    test_path = os.path.join(model_dir, f"test_{crop_mode}_results.npz")

    print("\n==================================================")
    print(f"Build matrix: {model_tag}/{crop_mode}")
    print("train_path:", train_path)
    print("test_path:", test_path)
    print("fusion_name:", fusion_name)
    print("knn_weight:", fusion["knn"])
    print("logit_weight:", fusion["logit"])
    print("proto_weight:", fusion["proto"])
    print("proto_mode:", PROTO_MODE)
    print("==================================================")

    if not os.path.exists(train_path):
        raise FileNotFoundError(train_path)
    if not os.path.exists(test_path):
        raise FileNotFoundError(test_path)

    train_results = np.load(train_path, allow_pickle=True)
    test_results = np.load(test_path, allow_pickle=True)

    train_label = train_results["label"]

    knn_mat = knn_both_feat(
        NUM_CLASSES,
        train_results["embed_features1"],
        train_results["embed_features2"],
        test_results["embed_features1"],
        test_results["embed_features2"],
        train_label,
    )

    logit_mat = restore_all_pred(
        NUM_CLASSES,
        test_results["pred_logit"],
        test_results["pred_idx"],
    )

    if USE_PROTO_SCORE:
        if PROTO_MODE != "single":
            raise NotImplementedError(
                "This code implements proto_mode='single' only."
            )

        proto_mat = prototype_all_pred_torch(
            NUM_CLASSES,
            train_results["embed_features1"],
            train_results["embed_features2"],
            test_results["embed_features1"],
            test_results["embed_features2"],
            train_label,
            chunk_size=KNN_CHUNK_SIZE,
        )

        print("Combining KNN/logit/proto matrices...", flush=True)

        mat = (
            knn_weight * knn_mat
            + logit_weight * logit_mat
            + proto_weight * proto_mat
        )

        print("Combined matrix:", mat.shape, mat.dtype, flush=True)
        print("Saving matrix cache...", flush=True)

        os.makedirs(os.path.dirname(cache_mat_path), exist_ok=True)
        np.save(cache_mat_path, mat.astype("float32"))

        print("Saved matrix cache:", cache_mat_path, flush=True)

        del proto_mat

    else:
        mat = knn_weight * knn_mat + logit_weight * logit_mat

    mat = apply_species_bonus_to_mat(
        mat=mat,
        train_results=train_results,
        test_results=test_results,
        n_class=NUM_CLASSES,
        species_bonus=species_bonus,
    )

    test_images = test_results["image"]

    np.save(cache_mat_path, mat.astype("float32"))
    np.save(cache_img_path, test_images)

    print("Saved matrix:", cache_mat_path)
    print("Saved test images:", cache_img_path)

    del train_results, test_results, knn_mat, logit_mat
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return mat.astype("float32"), test_images


def build_weighted_crop_matrix(
    model_tag,
    crop_weights,
    crop_name,
    fusion,
    species_bonus,
):
    fusion_name = fusion["name"]
    """
    Build model-level weighted crop matrix.
    """
    save_name = (
        f"{model_tag}_{crop_name}"
        f"_{fusion_name}"
        f"_species{species_bonus:.2f}_model_mat.npy"
    )
    save_img_name = save_name.replace("_model_mat.npy", "_test_images.npy")

    save_mat_path = os.path.join(CACHE_DIR, save_name)
    save_img_path = os.path.join(CACHE_DIR, save_img_name)

    # ============================================================
    # Fast mode:
    # Use precomputed model-level matrix from local cache or Hugging Face.
    # This skips KNN / logit / proto rebuilding.
    # ============================================================
    if USE_HF_MODEL_MATS:
        if not os.path.exists(save_mat_path) or not os.path.exists(
            save_img_path
        ):
            download_model_mat_from_hf_if_missing(save_mat_path)
            download_model_mat_from_hf_if_missing(save_img_path)

        print("Loading HF/local model matrix:", save_mat_path)

        model_mat = np.load(save_mat_path, mmap_mode="r")
        test_images = np.load(save_img_path, allow_pickle=True)

        if model_mat.shape[1] != NUM_CLASSES:
            raise ValueError(
                f"{save_mat_path} class dim {model_mat.shape[1]} != NUM_CLASSES {NUM_CLASSES}"
            )

        if model_mat.shape[0] != len(test_images):
            raise ValueError(
                f"{save_mat_path} rows {model_mat.shape[0]} != test_images {len(test_images)}"
            )

        print("Loaded model_mat:", model_mat.shape, model_mat.dtype)
        print("Loaded test_images:", test_images.shape)

        return model_mat, test_images

    # ============================================================
    # Normal local cache:
    # Use local model matrix if it already exists.
    # ============================================================
    if (
        (not FORCE_REBUILD)
        and os.path.exists(save_mat_path)
        and os.path.exists(save_img_path)
    ):
        print("Loading cached weighted crop matrix:", save_mat_path)
        return (
            np.load(save_mat_path, mmap_mode="r"),
            np.load(save_img_path, allow_pickle=True),
        )

    print("\n========================================")
    print("Build weighted crop matrix:", model_tag, crop_name)
    print("crop_weights:", crop_weights)
    print("fusion_name:", fusion_name)
    print("knn_weight:", fusion["knn"])
    print("logit_weight:", fusion["logit"])
    print("proto_weight:", fusion["proto"])
    print("proto_mode:", PROTO_MODE)
    print("species_bonus:", species_bonus)
    print("========================================")

    total_w = sum(crop_weights.values())
    assert abs(total_w - 1.0) < 1e-6

    out = None
    test_images_ref = None

    for crop_mode, w in crop_weights.items():
        crop_mat, test_images = build_or_load_crop_matrix(
            model_tag=model_tag,
            crop_mode=crop_mode,
            fusion=fusion,
            species_bonus=species_bonus,
        )

        if test_images_ref is None:
            test_images_ref = test_images
        else:
            assert np.array_equal(test_images_ref, test_images), (
                f"Test image order mismatch: {model_tag}/{crop_mode}"
            )

        if out is None:
            out = crop_mat.astype("float32") * w
        else:
            out += crop_mat.astype("float32") * w

        del crop_mat
        gc.collect()

    np.save(save_mat_path, out.astype("float32"))
    np.save(save_img_path, test_images_ref)

    print("Saved weighted crop matrix:", save_mat_path)
    print("Saved weighted crop test images:", save_img_path)

    upload_model_mat_to_hf_if_enabled(save_mat_path)
    upload_model_mat_to_hf_if_enabled(save_img_path)

    print(
        "Uploaded model matrix to HF:",
        f"model_mats/{os.path.basename(save_mat_path)}",
    )
    print(
        "Uploaded test images to HF:",
        f"model_mats/{os.path.basename(save_img_path)}",
    )

    print("Saved weighted crop matrix:", save_mat_path)

    return out.astype("float32"), test_images_ref


# ============================================================
# Run full ensemble
# ============================================================

for species_bonus in SPECIES_BONUS_LIST:
    for fusion in FUSION_WEIGHT_SETS:
        fusion_name = fusion["name"]

        print(
            "\n\n############################################################"
        )
        print("RUN SETTING")
        print("species_bonus:", species_bonus)
        print("fusion_name:", fusion_name)
        print("knn_weight:", fusion["knn"])
        print("logit_weight:", fusion["logit"])
        print("proto_weight:", fusion["proto"])
        print("proto_mode:", PROTO_MODE)
        print("############################################################")

        v2xl_mat, test_images_v2xl = build_weighted_crop_matrix(
            model_tag="v2xl",
            crop_weights=V2XL_CROP_WEIGHTS,
            crop_name="v2xl_fb080_bf010_none010",
            fusion=fusion,
            species_bonus=species_bonus,
        )

        b7_mat, test_images_b7 = build_weighted_crop_matrix(
            model_tag="b7",
            crop_weights=B7_CROP_WEIGHTS,
            crop_name="b7pseudo_fb085_bf005_none010",
            fusion=fusion,
            species_bonus=species_bonus,
        )

        b6_mat, test_images_b6 = build_weighted_crop_matrix(
            model_tag="b6",
            crop_weights=B6_CROP_WEIGHTS,
            crop_name="b6charm_fb060_charm025_bf005_none010",
            fusion=fusion,
            species_bonus=species_bonus,
        )

        conv_mat, test_images_conv = build_weighted_crop_matrix(
            model_tag="conv",
            crop_weights=CONV_CROP_WEIGHTS,
            crop_name="convpseudo_fb085_bf005_none010",
            fusion=fusion,
            species_bonus=species_bonus,
        )

        assert np.array_equal(test_images_v2xl, test_images_b7), (
            "V2XL and B7 test image order mismatch!"
        )
        assert np.array_equal(test_images_v2xl, test_images_b6), (
            "V2XL and B6 test image order mismatch!"
        )
        assert np.array_equal(test_images_v2xl, test_images_conv), (
            "V2XL and Conv test image order mismatch!"
        )

        for mw in MODEL_WEIGHT_SETS:
            name = mw["name"]

            v2xl_w = mw["v2xl"]
            b7_w = mw["b7"]
            b6_w = mw["b6"]
            conv_w = mw["conv"]

            assert abs(v2xl_w + b7_w + b6_w + conv_w - 1.0) < 1e-6

            print("V2XL:", v2xl_w, "B7:", b7_w, "B6:", b6_w, "Conv:", conv_w)

            ens_mat = (
                v2xl_w * v2xl_mat.astype("float32")
                + b7_w * b7_mat.astype("float32")
                + b6_w * b6_mat.astype("float32")
                + conv_w * conv_mat.astype("float32")
            )

            out_prefix = (
                f"ensemble_v2xl_b7pseudo_b6charm_convpseudo"
                f"_{fusion_name}"
                f"_species{species_bonus:.2f}"
                f"_{name}"
            )

            make_submission_from_matrix(
                ens_mat,
                test_images_b7,
                out_prefix=out_prefix,
            )

            del ens_mat
            gc.collect()

        del v2xl_mat, b7_mat, b6_mat, conv_mat
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("Done.")
print("Output dir:", OUT_DIR)
print(os.listdir(OUT_DIR))

# hf_token

In [ ]:
# ============================================================
# Auto disconnect Colab runtime when all jobs are done
# Put this cell at the VERY END of the notebook
# ============================================================

import time

AUTO_DISCONNECT = True

if AUTO_DISCONNECT:
    print("All tasks finished.")
    print("Colab runtime will disconnect in 60 seconds.")
    print("Stop this cell now if you want to cancel auto disconnect.")

    time.sleep(60)

    try:
        from google.colab import runtime

        runtime.unassign()
    except Exception as e:
        print("runtime.unassign() failed:", e)
        print("Fallback: killing current process.")
        import os

        os.kill(os.getpid(), 9)
else:
    print("AUTO_DISCONNECT = False, keep runtime alive.")